<a href="https://colab.research.google.com/github/jameshphan-png/Coding-Exercise---Prompt-Engineering/blob/dev/Prompt_Engineering_Part_3_(Self_Reflection).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Self-Reflection - (Tools used: ClaudeAI & api.llm7)

#Prompt: I want you to take into the shoes of a customer who is completely frustrated and unsatisfied with their order.
# After realizing that they cannot return their product because it is defective, they provide feedback, giving it a very low rating to the support system.
# How would you react in this case, in order to satisfy the customer once again, or reconcile the "relationship" that was damaged, given that their product is non-returnable?

In [ ]:
import subprocess
subprocess.run(["pip", "install", "openai", "-q"], check=True)

import os
import re
import json
from datetime import datetime, timedelta
from openai import OpenAI

# ─── Free AI Client (No API Key Required) ────────────────────────────────────
client = OpenAI(
    base_url="https://api.llm7.io/v1",
    api_key="unused"
)

MODEL = "gpt-4o-mini-2024-07-18"


# ─── Company Configuration ────────────────────────────────────────────────────
COMPANY_CONFIG = {
    "name": "Acme Corp",
    "industry": "E-commerce",
    "support_email": "support@acmecorp.com",
    "support_hours": "Monday-Friday, 9 AM - 6 PM EST",
    "website": "https://www.acmecorp.com",
    "return_policy": "30-day hassle-free returns",
}


# ─── Category Menus & Guided Forms ────────────────────────────────────────────
CATEGORIES = {
    "1": "Orders & Shipping",
    "2": "Returns & Refunds",
    "3": "Billing & Payments",
    "4": "Technical Support",
    "5": "Account Help",
    "6": "General Inquiry",
}

GUIDED_FORMS = {
    "Orders & Shipping": [
        ("order_number",  "What is your order number? (e.g. ORD-12345 or 12345)"),
        ("issue",         "What's the issue?\n   1. I haven't received my order\n   2. My order arrived damaged\n   3. I received the wrong item\n   4. I need to change my delivery address\n   Enter 1-4: "),
        ("contact_email", "What email is on your account?"),
    ],
    "Returns & Refunds": [
        ("order_number",  "What is your order number? (e.g. ORD-12345 or 12345)"),
        ("return_reason", "Why are you returning?\n   1. Item is defective\n   2. Wrong item received\n   3. Changed my mind\n   4. Item not as described\n   Enter 1-4: "),
        ("preference",    "Would you prefer a refund or exchange? (type refund or exchange): "),
        ("contact_email", "What email is on your account?"),
    ],
    "Billing & Payments": [
        ("issue",         "What's your billing issue?\n   1. I was charged incorrectly\n   2. My payment was declined\n   3. I need a copy of my invoice\n   4. I want to update my payment method\n   Enter 1-4: "),
        ("order_number",  "Related order number? (press Enter to skip): "),
        ("contact_email", "What email is on your account?"),
    ],
    "Technical Support": [
        ("platform",      "Where are you experiencing the issue?\n   1. Website\n   2. Mobile App\n   3. Account login\n   4. Other\n   Enter 1-4: "),
        ("description",   "Briefly describe the problem you're experiencing: "),
        ("contact_email", "What email can we reach you at?"),
    ],
    "Account Help": [
        ("issue",         "What do you need help with?\n   1. I can't log in\n   2. I want to update my details\n   3. I want to delete my account\n   4. I didn't receive a verification email\n   Enter 1-4: "),
        ("contact_email", "What email is on your account?"),
    ],
    "General Inquiry": [
        ("topic",         "What would you like to know about? (briefly describe): "),
        ("contact_email", "What email can we reach you at?"),
    ],
}

SUB_LABELS = {
    "Orders & Shipping": {
        "1": "hasn't received order", "2": "order arrived damaged",
        "3": "received wrong item",   "4": "needs to change delivery address",
    },
    "Returns & Refunds": {
        "1": "item is defective",  "2": "wrong item received",
        "3": "changed their mind", "4": "item not as described",
    },
    "Billing & Payments": {
        "1": "charged incorrectly", "2": "payment was declined",
        "3": "needs invoice copy",  "4": "wants to update payment method",
    },
    "Technical Support": {
        "1": "website issue", "2": "mobile app issue",
        "3": "account login", "4": "other technical issue",
    },
    "Account Help": {
        "1": "can't log in",            "2": "wants to update details",
        "3": "wants to delete account", "4": "didn't receive verification email",
    },
}


# ─── Mock "Databases" & "APIs" ───────────────────────────────────────────────
TODAY = datetime.now()

ORDER_DB = {
    "ORD-12345": {
        "order_number": "ORD-12345",
        "delivered_at": (TODAY - timedelta(days=10)).strftime("%Y-%m-%d"),
        "status": "delivered",
        "item": "Wireless Headphones",
        "price_usd": 89.99,
        "returnable": True,
    },
    "ORD-31892": {
        "order_number": "ORD-31892",
        "delivered_at": (TODAY - timedelta(days=8)).strftime("%Y-%m-%d"),
        "status": "delivered",
        "item": "Bluetooth Speaker",
        "price_usd": 49.99,
        "returnable": True,
    },
    "ORD-54321": {
        "order_number": "ORD-54321",
        "delivered_at": (TODAY - timedelta(days=45)).strftime("%Y-%m-%d"),
        "status": "delivered",
        "item": "Standing Desk",
        "price_usd": 249.00,
        "returnable": False,
    },
}

POLICY_DB = {
    "returns_policy": {
        "window_days": 30,
        "condition": "unused items in original packaging",
        "refund_timeline_days": "5–10 business days after inspection",
        "refund_method": "original payment method",
        "shipping_fee_refund": "Depends on reason; damaged/wrong item usually covered.",
    },
    "return_options": [
        {
            "name": "Prepaid return label (drop-off)",
            "details": "We email a prepaid label. Drop off at partner carrier locations.",
            "typical_time": "2–7 days transit to warehouse, then inspection.",
        },
        {
            "name": "Scheduled pickup (where available)",
            "details": "We arrange a carrier pickup at your address (may have a small fee).",
            "typical_time": "1–3 days to pickup + transit time.",
        },
        {
            "name": "In-store return (if purchased online + store near you)",
            "details": "Bring the item and order number. Immediate acceptance; refund still follows timeline.",
            "typical_time": "Same day acceptance, refund timeline still applies.",
        },
    ],
}


def normalize_order_number(order_number: str) -> str:
    if not order_number:
        return ""
    s = order_number.strip().upper()
    if re.match(r"^ORD-\d+$", s):
        return s
    digits = re.findall(r"\d+", s)
    if digits:
        return f"ORD-{digits[0]}"
    return s


def search_policies(topic: str) -> dict:
    topic = (topic or "").lower()
    if "refund" in topic or "return" in topic:
        return {
            "found": True,
            "policy": POLICY_DB["returns_policy"],
            "options": POLICY_DB["return_options"],
        }
    return {"found": False, "message": "No policy found for that topic."}


def check_order(order_number: str) -> dict:
    normalized = normalize_order_number(order_number)
    if not normalized:
        return {"found": False, "message": "No order number provided."}
    rec = ORDER_DB.get(normalized)
    if not rec:
        return {"found": False, "message": f"Order not found for '{normalized}'."}
    return {"found": True, "order": rec}


def compute_return_eligibility(order: dict) -> dict:
    delivered_at = datetime.strptime(order["delivered_at"], "%Y-%m-%d")
    days_since = (TODAY - delivered_at).days
    window = POLICY_DB["returns_policy"]["window_days"]
    eligible = days_since <= window and order.get("returnable", True)
    return {
        "eligible": eligible,
        "days_since_delivery": days_since,
        "window_days": window,
        "reason": "Within return window." if eligible else "Outside return window or item non-returnable.",
    }


def generate_return_instructions(order: dict, preference: str, reason: str) -> dict:
    preference = (preference or "").strip().lower()
    reason = (reason or "").strip().lower()

    steps = [
        "Confirm the item is in the original packaging and unused (if possible).",
        "We'll email return instructions and a label to the contact email on file.",
        "Pack the item securely and attach the label.",
        "Drop off at the carrier (or request pickup if available).",
        "After inspection, refunds are processed to the original payment method.",
    ]

    if preference == "exchange":
        steps.insert(0, "We can process an exchange once the return is received and inspected.")

    if "defective" in reason or "wrong item" in reason:
        fee_note = "Return shipping is typically covered for defective or wrong-item cases."
    else:
        fee_note = "Return shipping may be deducted depending on the return reason."

    return {
        "steps": steps,
        "fee_note": fee_note,
        "refund_timeline": POLICY_DB["returns_policy"]["refund_timeline_days"],
        "refund_method": POLICY_DB["returns_policy"]["refund_method"],
    }


# ─── ReACT System Prompt ──────────────────────────────────────────────────────
# FIX: The original prompt's instruction to respond "ONLY as valid JSON" caused
# the agent to output {"type":"final","message":"Thanks—how can I help?"} without
# actually running any tools first. The new prompt explicitly instructs the agent
# to ALWAYS use tools before producing a final message, especially on the first turn.
REACT_SYSTEM_PROMPT = f"""
You are a friendly and professional customer support agent for {COMPANY_CONFIG["name"]} ({COMPANY_CONFIG["industry"]}).

You use a ReACT loop: always take tool actions BEFORE giving a final answer.
You MUST respond ONLY as valid JSON (no extra text outside the JSON object).

Available tools (actions):
1) search_policies: input={{"topic": "<string>"}}
2) check_order: input={{"order_number": "<string>"}}
3) compute_return_eligibility: input={{"order": <object from check_order>}}
4) generate_return_instructions: input={{"order": <object>, "preference": "<refund|exchange>", "reason": "<string>"}}
5) escalate_to_human: input={{"reason":"<string>"}}

CRITICAL RULES — follow these every time:
- On the VERY FIRST turn, you MUST run tools before producing type=final. Never skip straight to final.
- For ANY returns/refunds request with an order number, ALWAYS run these steps in order:
    a) search_policies with topic="returns and refunds"
    b) check_order with the provided order_number
    c) compute_return_eligibility with the order object returned
    d) generate_return_instructions with the order, preference, and reason
- Only after running all relevant tools should you produce a type=final response.
- Your final message MUST reference the specific order, eligibility result, and next steps.
- If the order is NOT eligible (outside window or non-returnable), clearly explain why and offer to escalate.
- If key info is missing, ask a targeted clarification as type=final.
- If the case is outside policy or high-risk, use escalate_to_human first, then produce type=final.
- FINAL messages should be warm, concise (4–6 sentences), and include concrete next steps.

JSON response formats:
  Action:  {{"type": "action", "name": "<tool_name>", "input": {{...}} }}
  Final:   {{"type": "final",  "message": "<your response to the customer>" }}

Company details:
- Support email: {COMPANY_CONFIG["support_email"]}
- Support hours: {COMPANY_CONFIG["support_hours"]}
- Return policy: {COMPANY_CONFIG["return_policy"]}
"""


# ─── Conversation Logger ──────────────────────────────────────────────────────
class ConversationLogger:
    def __init__(self, log_file="support_log.json"):
        self.log_file = log_file
        self.session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.log_data = {
            "session_id": self.session_id,
            "started_at": datetime.now().isoformat(),
            "company": COMPANY_CONFIG["name"],
            "messages": [],
        }

    def log(self, role, content):
        self.log_data["messages"].append({
            "timestamp": datetime.now().isoformat(),
            "role": role,
            "content": content,
        })

    def save(self):
        self.log_data["ended_at"] = datetime.now().isoformat()
        try:
            existing = []
            if os.path.exists(self.log_file):
                with open(self.log_file, "r") as f:
                    existing = json.load(f)
            existing.append(self.log_data)
            with open(self.log_file, "w") as f:
                json.dump(existing, f, indent=2)
            print("\n Session saved to " + self.log_file + " (ID: " + self.session_id + ")")
        except Exception as e:
            print("\n Could not save log: " + str(e))


# ─── Helpers ─────────────────────────────────────────────────────────────────
def print_divider():
    print("-" * 58)

def show_main_menu():
    print_divider()
    print("  Please select a support category:\n")
    for key, label in CATEGORIES.items():
        print("    " + key + ". " + label)
    print_divider()

def collect_form(category):
    questions = GUIDED_FORMS[category]
    answers = {}
    print("\n  Let's gather some details about your " + category + " issue.\n")
    for field, prompt in questions:
        while True:
            answer = input("  " + prompt + " ").strip()
            if answer == "" and "skip" in prompt.lower():
                answers[field] = "N/A"
                break
            if answer:
                answers[field] = answer
                break
            print("  Please enter a value (or press Enter to skip if allowed).")
    return answers

def build_summary(category, answers):
    lines = ["Customer category: " + category]
    for field, value in answers.items():
        if field in ("issue", "return_reason", "platform") and value in SUB_LABELS.get(category, {}):
            value = SUB_LABELS[category][value]
        lines.append(field.replace("_", " ").capitalize() + ": " + value)
    return "\n".join(lines)


# ─── LLM Call ────────────────────────────────────────────────────────────────
def ask_ai(messages):
    try:
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=512,
            messages=messages,
            temperature=0.2,
        )
        return response.choices[0].message.content
    except Exception as e:
        msg = str(e)
        if "Error code: 429" in msg or "Rate limit exceeded" in msg:
            raise RuntimeError("RATE_LIMIT_429")
        raise


# ─── ReACT Agent Loop ────────────────────────────────────────────────────────
def safe_json_loads(s: str) -> dict:
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", s, re.S)
        if m:
            return json.loads(m.group(0))
        raise


def tool_dispatch(action_name: str, action_input: dict) -> dict:
    if action_name == "search_policies":
        return search_policies(action_input.get("topic", ""))
    if action_name == "check_order":
        return check_order(action_input.get("order_number", ""))
    if action_name == "compute_return_eligibility":
        order = action_input.get("order", {})
        if not order:
            return {"error": "Missing order object for eligibility check."}
        return compute_return_eligibility(order)
    if action_name == "generate_return_instructions":
        order = action_input.get("order", {})
        if not order:
            return {"error": "Missing order object for return instructions."}
        return generate_return_instructions(
            order=order,
            preference=action_input.get("preference", ""),
            reason=action_input.get("reason", ""),
        )
    if action_name == "escalate_to_human":
        return {
            "escalated": True,
            "reason": action_input.get("reason", "Requires human review."),
            "contact": COMPANY_CONFIG["support_email"],
            "sla": "within 1 business day",
        }
    return {"error": f"Unknown action: {action_name}"}


def local_refund_policy_answer() -> str:
    p = POLICY_DB["returns_policy"]
    options = POLICY_DB["return_options"]
    return (
        f"Our return window is {p['window_days']} days from delivery "
        f"(items should be {p['condition']}). "
        f"Refunds are issued to the {p['refund_method']} in {p['refund_timeline_days']}. "
        f"Return options: {options[0]['name']}, {options[1]['name']}, or {options[2]['name']}. "
        "Share your order number and I can confirm eligibility and next steps."
    )


def run_react_agent(user_summary: str, followup_messages: list, logger: ConversationLogger) -> str:
    messages = [{"role": "system", "content": REACT_SYSTEM_PROMPT}]
    messages.append({"role": "user", "content": user_summary})
    for m in followup_messages:
        messages.append(m)

    max_steps = 8
    for _ in range(max_steps):
        try:
            raw = ask_ai(messages)
        except RuntimeError as e:
            if str(e) == "RATE_LIMIT_429":
                last_user = next(
                    (mm.get("content", "") for mm in reversed(followup_messages) if mm.get("role") == "user"),
                    ""
                )
                if re.search(r"\b(refund|return)\b", last_user.lower()):
                    return local_refund_policy_answer()
                return (
                    "I'm temporarily rate-limited. Please try again in a minute, "
                    f"or email {COMPANY_CONFIG['support_email']} and we'll respond within 1 business day."
                )
            raise

        logger.log("assistant_raw", raw)

        try:
            obj = safe_json_loads(raw)
        except Exception:
            return (
                "Sorry, I ran into a formatting issue. "
                "Could you confirm your order number and whether you'd like a refund or exchange?"
            )

        if obj.get("type") == "final":
            return obj.get("message", "How can I help you further?")

        if obj.get("type") == "action":
            name = obj.get("name", "")
            action_input = obj.get("input", {}) or {}
            observation = tool_dispatch(name, action_input)

            logger.log("tool_call", {"name": name, "input": action_input})
            logger.log("tool_observation", observation)

            messages.append({"role": "assistant", "content": json.dumps({"type": "action", "name": name, "input": action_input})})
            messages.append({"role": "user", "content": json.dumps({"type": "observation", "name": name, "output": observation})})
            continue

        # Schema error — nudge the model
        messages.append({
            "role": "user",
            "content": json.dumps({
                "type": "observation",
                "name": "schema_error",
                "output": {"error": "You must output JSON with type=action or type=final."}
            })
        })

    return (
        "I'm going to connect you with a human agent to finish this. "
        f"Please email {COMPANY_CONFIG['support_email']} with your order number and request."
    )


# ─── FIX: Closing flow with satisfaction check ────────────────────────────────
# The original code exited immediately when the user typed "done" with only a
# hardcoded print statement. Now the agent produces a proper closing summary and
# asks for a satisfaction rating before ending the session.
def ask_satisfaction_rating() -> str:
    print("\n" + "-" * 58)
    print("  Before you go, how satisfied were you with your support today?")
    print("    1. Very satisfied")
    print("    2. Satisfied")
    print("    3. Neutral")
    print("    4. Unsatisfied")
    print("    5. Very unsatisfied")
    print("-" * 58)
    while True:
        rating = input("  Enter 1-5 (or press Enter to skip): ").strip()
        if rating == "":
            return "skipped"
        if rating in ("1", "2", "3", "4", "5"):
            labels = {
                "1": "Very satisfied", "2": "Satisfied", "3": "Neutral",
                "4": "Unsatisfied",    "5": "Very unsatisfied"
            }
            return labels[rating]
        print("  Please enter a number between 1 and 5.")


def print_closing_message(rating: str, contact_email: str):
    print()
    print_divider()
    if rating in ("Very satisfied", "Satisfied"):
        print("  Agent: So glad we could help! Your case is logged and you'll\n"
              "         receive a confirmation email shortly. Have a great day!")
    elif rating == "Neutral":
        print("  Agent: Thank you for your feedback. If you need anything else,\n"
              "         don't hesitate to reach out at " + contact_email + ".")
    elif rating in ("Unsatisfied", "Very unsatisfied"):
        print("  Agent: We're sorry the experience wasn't better. A senior agent\n"
              "         will follow up at your email address to make things right.")
    else:
        print("  Agent: Thank you for contacting " + COMPANY_CONFIG["name"] + " support. Have a great day!")
    print_divider()


# ─── Main Support Flow ────────────────────────────────────────────────────────
def run_support_session(logger):
    show_main_menu()

    while True:
        choice = input("  Enter number (1-6): ").strip()
        if choice in CATEGORIES:
            category = CATEGORIES[choice]
            print("\n  You selected: " + category)
            break
        print("  Please enter a number between 1 and 6.")

    answers = collect_form(category)
    summary = build_summary(category, answers)
    logger.log("user_form", summary)

    print("\n  Connecting you with our support agent...\n")
    print_divider()

    followups = []
    agent_reply = run_react_agent(summary, followups, logger)
    logger.log("assistant", agent_reply)

    print("Support Agent:\n\n  " + agent_reply + "\n")
    print_divider()

    print("  Need anything else? Type a follow-up question,")
    print("  or type 'done' when you're finished.\n")

    while True:
        follow_up = input("  You: ").strip()
        if not follow_up:
            continue

        # FIX: "done" now triggers the closing flow instead of a bare print
        if follow_up.lower() in ("done", "quit", "exit", "no", "nope"):
            rating = ask_satisfaction_rating()
            logger.log("satisfaction_rating", rating)
            print_closing_message(rating, COMPANY_CONFIG["support_email"])
            break

        followups.append({"role": "user", "content": follow_up})
        logger.log("user", follow_up)

        reply = run_react_agent(summary, followups, logger)
        followups.append({"role": "assistant", "content": reply})
        logger.log("assistant", reply)

        print("\nAgent: " + reply + "\n")


# ─── Entry Point ─────────────────────────────────────────────────────────────
def main():
    print("=" * 58)
    print("  " + COMPANY_CONFIG["name"] + " - Customer Support (ReACT Demo)")
    print("  " + COMPANY_CONFIG["support_hours"])
    print("  " + COMPANY_CONFIG["support_email"])
    print("=" * 58)

    while True:
        logger = ConversationLogger()
        try:
            run_support_session(logger)
        except Exception as e:
            print("\n  Something went wrong: " + str(e))
            print("  The session will now close safely.")
        finally:
            logger.save()

        again = input("\n  Start a new support session? (yes / no): ").strip().lower()
        if again not in ("yes", "y"):
            print("\n  Thank you for contacting " + COMPANY_CONFIG["name"] + " support. Have a great day!\n")
            break

main()

  Acme Corp - Customer Support (ReACT Demo)
  Monday-Friday, 9 AM - 6 PM EST
  support@acmecorp.com
----------------------------------------------------------
  Please select a support category:

    1. Orders & Shipping
    2. Returns & Refunds
    3. Billing & Payments
    4. Technical Support
    5. Account Help
    6. General Inquiry
----------------------------------------------------------
  Enter number (1-6): 2

  You selected: Returns & Refunds

  Let's gather some details about your Returns & Refunds issue.

  What is your order number? (e.g. ORD-12345 or 12345) Ord-54321
  Why are you returning?
   1. Item is defective
   2. Wrong item received
   3. Changed my mind
   4. Item not as described
   Enter 1-4:  1
  Would you prefer a refund or exchange? (type refund or exchange):  refund
  What email is on your account? jamesphan@gmail.com

  Connecting you with our support agent...

----------------------------------------------------------
Support Agent:

  Thank you for reac